# 使用 LeNet 进行手写数字识别 (MNIST)

我们将使用 LeNet 模型来识别 MNIST 手写数字数据集。这包括加载和预处理数据、训练模型以及评估其性能。

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
import matplotlib.pyplot as plt

# 检查 GPU 是否可用
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  0


## 1. LeNet 模型定义 (重新定义，以防前面的单元格被删除或修改)

这里再次给出 LeNet-5 模型的定义。它由两个卷积层、两个池化层和三个全连接层组成。

In [ ]:
class LeNet(Model):
    def __init__(self, num_classes=10):
        super(LeNet, self).__init__()
        # C1: 卷积层 (6个 5x5 卷积核, stride=1, padding='valid')
        # MNIST 图像是 28x28, LeNet 原论文输入是 32x32, 这里将使用 padding='same' 或在数据预处理时调整
        # 考虑到 MNIST 图像是 28x28，如果直接使用 5x5 valid padding，会迅速减小尺寸。
        # 为了更好地适应 28x28，我们对原始 LeNet 结构进行微调，或者在预处理时将图像填充到 32x32。
        # 这里我们选择在输入前进行填充，以保持模型结构与 LeNet 原始设计更接近。
        self.conv1 = layers.Conv2D(filters=6, kernel_size=(5, 5), activation='tanh', padding='valid')
        # S2: 平均池化层 (2x2 池化核, stride=2)
        self.pool2 = layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))
        # C3: 卷积层 (16个 5x5 卷积核, stride=1, padding='valid')
        self.conv3 = layers.Conv2D(filters=16, kernel_size=(5, 5), activation='tanh', padding='valid')
        # S4: 平均池化层 (2x2 池化核, stride=2)
        self.pool4 = layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2))
        # C5: 展平层
        self.flatten = layers.Flatten()
        # F6: 全连接层 (120个神经元)
        self.fc5 = layers.Dense(units=120, activation='tanh')
        # F7: 全连接层 (84个神经元)
        self.fc6 = layers.Dense(units=84, activation='tanh')
        # Output: 输出层 (num_classes个神经元, softmax激活函数)
        self.output_layer = layers.Dense(units=num_classes, activation='softmax')

    def call(self, inputs):
        x = self.conv1(inputs)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.pool4(x)
        x = self.flatten(x)
        x = self.fc5(x)
        x = self.fc6(x)
        return self.output_layer(x)

## 2. 数据加载和预处理

我们将加载 MNIST 数据集，并对其进行以下预处理：

-   将图像尺寸从 28x28 填充到 32x32（以匹配 LeNet 原始输入要求）。
-   将像素值归一化到 `[0, 1]` 范围。
-   将标签转换为独热编码。
-   为卷积层添加通道维度。

In [ ]:
# 加载 MNIST 数据集
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 数据预处理
# 1. 填充图像到 32x32 (LeNet 原始输入尺寸)
x_train = np.pad(x_train, ((0,0),(2,2),(2,2)), 'constant', constant_values=0)
x_test = np.pad(x_test, ((0,0),(2,2),(2,2)), 'constant', constant_values=0)

# 2. 归一化像素值到 [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 3. 为卷积层添加通道维度 (单通道灰度图像)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# 4. 将标签转换为独热编码
y_train = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=10)

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")

x_train shape: (60000, 32, 32, 1)
y_train shape: (60000, 10)
x_test shape: (10000, 32, 32, 1)
y_test shape: (10000, 10)


## 3. 模型编译与训练

我们将实例化 LeNet 模型，然后使用 `Adam` 优化器和 `categorical_crossentropy` 损失函数来编译它。最后，在训练集上训练模型。

In [ ]:
# 实例化 LeNet 模型
lenet_model = LeNet(num_classes=10)

# 编译模型
lenet_model.compile(optimizer='adam',
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

# 训练模型
# 建议使用 GPU 运行，否则可能需要较长时间
epochs = 10
batch_size = 128

history = lenet_model.fit(x_train, y_train,
                            batch_size=batch_size,
                            epochs=epochs,
                            validation_data=(x_test, y_test))

# 打印模型结构摘要
lenet_model.build(input_shape=(None, 32, 32, 1)) # 使用一个示例输入形状来构建模型
lenet_model.summary()

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 34s 69ms/step - accuracy: 0.8970 - loss: 0.3444 - val_accuracy: 0.9509 - val_loss: 0.1550
Epoch 2/10
468/469 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.9585 - loss: 0.1352

KeyboardInterrupt: 

## 4. 模型评估

训练完成后，我们将在测试集上评估模型的性能，并可视化训练过程中的准确率和损失。

In [ ]:
# 评估模型
loss, accuracy = lenet_model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# 绘制训练历史
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

## 5. 进行预测 (可选)

我们可以使用训练好的模型对测试集中的一些图像进行预测，并查看结果。

In [ ]:
predictions = lenet_model.predict(x_test)

# 显示前几个测试图像及其预测结果
num_images = 5
plt.figure(figsize=(10, 5))
for i in range(num_images):
    plt.subplot(1, num_images, i + 1)
    plt.imshow(x_test[i].reshape(32, 32), cmap='gray')
    plt.title(f"Pred: {np.argmax(predictions[i])}\nTrue: {np.argmax(y_test[i])}")
    plt.axis('off')
plt.tight_layout()
plt.show()

# 使用 PyTorch 实现标准 LeNet-5

我们将按照 Yann LeCun 的原始论文结构，使用 PyTorch 框架重新实现 LeNet-5 进行 MNIST 手写数字识别。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 1. 定义 LeNet-5 模型
标准 LeNet-5 使用 `tanh` 作为激活函数，`AveragePooling` 作为池化层。

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # C1: 输入 1 通道, 输出 6 通道, 卷积核 5x5
        self.c1 = nn.Conv2d(1, 6, kernel_size=5)
        # S2: 平均池化 2x2, 步长 2
        self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)
        # C3: 卷积 16 通道, 卷积核 5x5
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)
        # S4: 平均池化 2x2, 步长 2
        self.s4 = nn.AvgPool2d(kernel_size=2, stride=2)
        # C5: 卷积 120 通道, 卷积核 5x5 (实质上是全连接)
        self.c5 = nn.Conv2d(16, 120, kernel_size=5)
        # F6: 全连接层 84
        self.f6 = nn.Linear(120, 84)
        # Output: 全连接层 10
        self.output = nn.Linear(84, 10)

        self.tanh = nn.Tanh()

    def forward(self, x):
        x = self.tanh(self.c1(x))
        x = self.s2(x)
        x = self.tanh(self.c3(x))
        x = self.s4(x)
        x = self.tanh(self.c5(x))
        # 展平
        x = x.view(x.size(0), -1)
        x = self.tanh(self.f6(x))
        x = self.output(x)
        return x

model = LeNet5().to(device)
print(model)

### 2. 加载 MNIST 数据集
原始 LeNet 输入为 32x32，而 MNIST 为 28x28，因此我们需要进行 Padding。

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

### 3. 训练模型

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

train(epochs=5)

### 4. 测试模型

In [ ]:
def test():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Accuracy on test images: {100 * correct / total:.2f}%')

test()

# 我自己的尝试
### 引入库

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

## 1.LeNet 的初定义


In [ ]:
class LeNet(nn.Module):
  def __init__(self):
    super(LeNet,self).__init__()
    self.conv1=nn.Conv2d(in_channels=1,out_channels=6,kernel_size=5) #[6,28,28]
    self.pool1=nn.AvgPool2d(kernel_size=2,stride=2) #[6,14,14]
    self.conv2=nn.Conv2d(in_channels=6,out_channels=16,kernel_size=5) #[16,10,10]
    self.pool2=nn.AvgPool2d(kernel_size=2,stride=2) #[16,5,5]
    self.conv3=nn.Conv2d(in_channels=16,out_channels=120,kernel_size=5) #[120,1,1]
    self.fc1=nn.Linear(in_features=120,out_features=84) #[84,1,1]
    self.fc2=nn.Linear(in_features=84,out_features=10) #[10,1,1]
  def forward(self,x):
    x=F.tanh(self.conv1(x)) #第一层卷积层，z->a
    x=self.pool1(x) #第一层池化层，a->a
    x=F.tanh(self.conv2(x)) #第二层卷积层 z->a
    x=self.pool2(x) #第二层池化层 a->a
    x=F.tanh(self.conv3(x)) #第三层卷积层 z->a
    x=x.view(-1,120) #展平
    x=F.tanh(self.fc1(x)) #第一层全连接层 z->a
    x=self.fc2(x)
    return x



## 2.数据集操作的预处理

In [ ]:
transform=transforms.Compose([
      transforms.Resize((32,32)),
      transforms.ToTensor(),
      transforms.Normalize((0.5,),(0.5))
    ])


## 3.数据集的下载

In [ ]:
train_dataset=datasets.MNIST(root='./data',train=True,transform=transform,download=True)
test_dataset=datasets.MNIST(root='./data',train=False,transform=transform,download=True)
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64,shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 37.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.07MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.07MB/s]


### 好奇dataset和loader的形状

In [ ]:
print(f"train_dataset 的样本数量: {len(train_dataset)}")
print(f"train_dataset 的类型：{type(train_dataset)}")
# 获取 train_dataset 中单个样本的形状
# train_dataset 中的每个元素是一个 (image, label) 对
sample_image, sample_label = train_dataset[0]
print(f"train_dataset 中单个图像的形状: {sample_image.shape}")
print(f"train_dataset 中单个标签的形状: {torch.tensor(sample_label).shape}")

train_dataset 的样本数量: 60000
train_dataset 的类型：<class 'torchvision.datasets.mnist.MNIST'>
train_dataset 中单个图像的形状: torch.Size([1, 32, 32])
train_dataset 中单个标签的形状: torch.Size([])


In [ ]:
# 获取 train_loader 中一个批次的形状
# 迭代 train_loader 可以得到 (batch_images, batch_labels)
batch_images, batch_labels = next(iter(train_loader))
print(f"train_loader 中图像批次的形状: {batch_images.shape}")
print(f"train_loader 中标签批次的形状: {batch_labels.shape}")

train_loader 中图像批次的形状: torch.Size([64, 1, 32, 32])
train_loader 中标签批次的形状: torch.Size([64])


## 4.初始化模型

In [ ]:
model=LeNet()
criterion=nn.CrossEntropyLoss()
optimizer=optim.SGD(model.parameters(),lr=0.1)
print(model)

LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv3): Conv2d(16, 120, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=120, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=10, bias=True)
)


## 5.训练过程

In [ ]:
def train(model,train_loader,criterion,optimizer,epochs=10):
  model.train()
  for epoch in range(epochs):
    running_loss=0.0
    for images,labels in train_loader:
      optimizer.zero_grad()
      output=model(images)
      loss=criterion(output,labels)
      loss.backward()
      optimizer.step()
      running_loss+=loss.item()
    print(f'Epoch:{epoch+1}, Loss:{running_loss/len(train_loader):.4f}')

## 6.测试过程

In [ ]:
def test(model,test_loader):
  model.eval()
  correct=0
  total=0
  with torch.no_grad():
    for images,labels in test_loader:
      output=model(images)
      _,predict=torch.max(output,1)
      total+=labels.size(0)
      correct+=(predict==labels).sum().item()
    print(f'correct rates:{100*correct/total:.2f}%')


In [ ]:
if __name__=='__main__':
  train(model,train_loader,criterion,optimizer,epochs=5)
  test(model,test_loader)

Epoch:1, Loss:0.0378
Epoch:2, Loss:0.0358
Epoch:3, Loss:0.0289
Epoch:4, Loss:0.0267
Epoch:5, Loss:0.0255
correct rates:98.45%


In [ ]:
# 实验：观察不同变量的类型
print(f"images 的类型: {type(batch_images)}")
print(f"labels 的类型: {type(batch_labels)}")

# 观察模型参数的类型
for name, param in model.named_parameters():
    print(f"参数 {name} 的类型: {type(param)}")
    break # 只看第一个层示例

# 观察 sum() 后的结果
count_tensor = (batch_labels == batch_labels).sum()
count_scalar = count_tensor.item()
print(f"sum() 后的类型: {type(count_tensor)}")
print(f"item() 后的类型: {type(count_scalar)}")

images 的类型: <class 'torch.Tensor'>
labels 的类型: <class 'torch.Tensor'>
参数 conv1.weight 的类型: <class 'torch.nn.parameter.Parameter'>
sum() 后的类型: <class 'torch.Tensor'>
item() 后的类型: <class 'int'>


# 实验对比：多层感知机 (MLP) 版本
在此部分，我们使用全连接神经网络（不含卷积层）来解决同样的 MNIST 分类问题。

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        # 输入：32*32 = 1024
        self.fc1 = nn.Linear(32 * 32, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        # 展平图像: [batch, 1, 32, 32] -> [batch, 1024]
        x = x.view(-1, 32 * 32)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 初始化 MLP 模型、损失函数和优化器
mlp_model = MLP()
mlp_criterion = nn.CrossEntropyLoss()
mlp_optimizer = optim.SGD(mlp_model.parameters(), lr=0.1)

print("MLP 模型结构:")
print(mlp_model)

MLP 模型结构:
MLP(
  (fc1): Linear(in_features=1024, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
)


### 训练与评估 MLP
我们使用与 LeNet 相同的超参数进行训练。

In [ ]:
print("开始训练 MLP...")
train(mlp_model, train_loader, mlp_criterion, mlp_optimizer, epochs=5)

print("\nMLP 在测试集上的表现:")
test(mlp_model, test_loader)

开始训练 MLP...
Epoch:1, Loss:0.4172
Epoch:2, Loss:0.1597
Epoch:3, Loss:0.1127
Epoch:4, Loss:0.0898
Epoch:5, Loss:0.0740

MLP 在测试集上的表现:
correct rates:96.99%


### 终极对比实验：LeNet vs MLP (均为 Batch Size = 1)
我们将观察 LeNet 在极端不稳定的梯度下，是否依然能靠其卷积结构提取出有效的特征。

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

# 1. 重新准备 Batch Size = 1 的数据加载器
sgd_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

# 2. 初始化两个用于对比的模型
sgd_lenet = LeNet()
sgd_mlp = MLP()

# 使用相同的优化器参数
sgd_lenet_optimizer = optim.SGD(sgd_lenet.parameters(), lr=0.01)
sgd_mlp_optimizer = optim.SGD(sgd_mlp.parameters(), lr=0.01)

criterion = torch.nn.CrossEntropyLoss()

# 3. 开始对比训练 (仅运行 1000 个 iteration)
print("正在以 Batch Size = 1 训练...")
for i, (image, label) in enumerate(sgd_loader):
    # 训练 LeNet
    sgd_lenet_optimizer.zero_grad()
    out_lenet = sgd_lenet(image)
    loss_lenet = criterion(out_lenet, label)
    loss_lenet.backward()
    sgd_lenet_optimizer.step()

    # 训练 MLP
    sgd_mlp_optimizer.zero_grad()
    out_mlp = sgd_mlp(image)
    loss_mlp = criterion(out_mlp, label)
    loss_mlp.backward()
    sgd_mlp_optimizer.step()

    if i % 250 == 0:
        print(f"Iteration {i}: LeNet Loss={loss_lenet.item():.4f}, MLP Loss={loss_mlp.item():.4f}")
    if i >= 1000: break

print("\n--- 最终实验对比结果 (训练 1000 个样本后) ---")
print("1. LeNet (Batch Size = 64, 5 Epochs) [刚才运行的]: 98.45%")
print("2. MLP (Batch Size = 64, 5 Epochs) [刚才运行的]: 96.99%")

print("\n接下来进行当前 BS=1 模型的即时评估:")
print("3. LeNet (Batch Size = 1) 测试准确率:")
test(sgd_lenet, test_loader)

print("4. MLP (Batch Size = 1) 测试准确率:")
test(sgd_mlp, test_loader)

正在以 Batch Size = 1 训练...
Iteration 0: LeNet Loss=2.2925, MLP Loss=2.3056
Iteration 250: LeNet Loss=2.1119, MLP Loss=1.1222
Iteration 500: LeNet Loss=1.1827, MLP Loss=0.4860
Iteration 750: LeNet Loss=0.7566, MLP Loss=0.4776

--- 最终实验对比结果 (训练 1000 个样本后) ---
1. LeNet (Batch Size = 64, 5 Epochs) [刚才运行的]: 98.45%
2. MLP (Batch Size = 64, 5 Epochs) [刚才运行的]: 96.99%

接下来进行当前 BS=1 模型的即时评估:
3. LeNet (Batch Size = 1) 测试准确率:
correct rates:86.31%
4. MLP (Batch Size = 1) 测试准确率:
correct rates:88.15%


## 更底层的 LeNet 实现 (不使用 `torch.nn` 模块)

### 1. 手动实现核心算子
我们将手动实现 2D 卷积和平均池化。卷积将通过滑动窗口和点积来实现。

In [1]:
import torch
import torch.nn.functional as F

def conv2d_low_level(x, weight, bias, stride=1, padding=0):
    # x: [batch, in_channels, h, w]
    # weight: [out_channels, in_channels, k_h, k_w]
    # bias: [out_channels]
    if padding > 0:
        x = F.pad(x, (padding, padding, padding, padding))

    batch, in_c, in_h, in_w = x.shape
    out_c, _, k_h, k_w = weight.shape

    out_h = (in_h - k_h) // stride + 1
    out_w = (in_w - k_w) // stride + 1

    # 初始化输出
    out = torch.zeros((batch, out_c, out_h, out_w), device=x.device)

    # 模拟卷积过程 (为了底层演示，使用循环)
    for i in range(out_h):
        for j in range(out_w):
            # 提取窗口
            window = x[:, :, i*stride:i*stride+k_h, j*stride:j*stride+k_w]
            # 执行卷积 (点乘求和)
            # window: [batch, in_c, k_h, k_w], weight: [out_c, in_c, k_h, k_w]
            # 利用广播机制计算
            for c_out in range(out_c):
                out[:, c_out, i, j] = torch.sum(window * weight[c_out], dim=(1, 2, 3)) + bias[c_out]
    return out

def avg_pool2d_low_level(x, kernel_size=2, stride=2):
    batch, c, h, w = x.shape
    out_h = (h - kernel_size) // stride + 1
    out_w = (w - kernel_size) // stride + 1
    out = torch.zeros((batch, c, out_h, out_w), device=x.device)

    for i in range(out_h):
        for j in range(out_w):
            window = x[:, :, i*stride:i*stride+kernel_size, j*stride:j*stride+kernel_size]
            out[:, :, i, j] = torch.mean(window, dim=(2, 3))
    return out

### 2. 定义底层 LeNet 类
我们不再继承 `nn.Module` 的自动参数管理，而是手动初始化 `torch.Tensor` 并设置 `requires_grad=True`。

In [2]:
class LowLevelLeNet:
    def __init__(self):
        # 手动初始化权重 (Kaiming 或 Xavier 初始化)
        self.params = {
            'c1_w': torch.randn(6, 1, 5, 5) * 0.1,
            'c1_b': torch.zeros(6),
            'c3_w': torch.randn(16, 6, 5, 5) * 0.1,
            'c3_b': torch.zeros(16),
            'c5_w': torch.randn(120, 16, 5, 5) * 0.1,
            'c5_b': torch.zeros(120),
            'f6_w': torch.randn(84, 120) * 0.1,
            'f6_b': torch.zeros(84),
            'out_w': torch.randn(10, 84) * 0.1,
            'out_b': torch.zeros(10)
        }
        # 启用梯度
        for p in self.params.values():
            p.requires_grad = True

    def forward(self, x):
        # C1: 5x5 conv -> tanh
        x = conv2d_low_level(x, self.params['c1_w'], self.params['c1_b'])
        x = torch.tanh(x)
        # S2: 2x2 avg pool
        x = avg_pool2d_low_level(x)

        # C3: 5x5 conv -> tanh
        x = conv2d_low_level(x, self.params['c3_w'], self.params['c3_b'])
        x = torch.tanh(x)
        # S4: 2x2 avg pool
        x = avg_pool2d_low_level(x)

        # C5: 5x5 conv -> tanh
        x = conv2d_low_level(x, self.params['c5_w'], self.params['c5_b'])
        x = torch.tanh(x)

        # Flatten
        x = x.view(x.size(0), -1)

        # F6: Linear -> tanh
        x = x @ self.params['f6_w'].t() + self.params['f6_b']
        x = torch.tanh(x)

        # Output: Linear
        x = x @ self.params['out_w'].t() + self.params['out_b']
        return x

# 实例化
low_level_model = LowLevelLeNet()
print("底层 LeNet 已初始化，参数已手动创建。")

底层 LeNet 已初始化，参数已手动创建。


### 3. 手动训练循环 (不使用 Optimizer)
我们将手动执行梯度下降：`parameter = parameter - learning_rate * gradient`。

In [5]:
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

# 确保数据集已加载
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

def train_low_level(model, data_loader, lr=0.01, iterations=10):
    print(f"开始底层训练演示 (仅运行 {iterations} 个迭代)...")
    for i, (images, labels) in enumerate(data_loader):
        if i >= iterations: break

        # 1. 前向传播
        outputs = model.forward(images)

        # 2. 计算损失 (Cross Entropy)
        loss = F.cross_entropy(outputs, labels)

        # 3. 反向传播 (生成梯度)
        loss.backward()

        # 4. 手动更新参数 (SGD)
        with torch.no_grad():
            for name, param in model.params.items():
                param -= lr * param.grad
                # 手动清空梯度
                param.grad.zero_()

        if i % 2 == 0:
            print(f"Iteration {i}, Loss: {loss.item():.4f}")

# 由于手写卷积较慢，我们用极小的 batch 测试一下逻辑
small_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
train_low_level(low_level_model, small_loader, lr=0.01, iterations=6)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 483kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.56MB/s]


开始底层训练演示 (仅运行 6 个迭代)...
Iteration 0, Loss: 2.3289
Iteration 2, Loss: 2.3569
Iteration 4, Loss: 2.8817
